In [1]:
import torch 
from einops import rearrange

In [2]:
plan_length = 4 # length of each trajectory (squence of 4 actions): T 
action_dim = 2  # action dimension : A
max_std = 2     
num_samples = 3 # sampling three different candidate plans : B 
num_elites = 2 
temperature = 0.005
n_iters = 7 

In [3]:
torch.manual_seed(123)

# action initialization
mean = torch.zeros(plan_length, action_dim) # (4,2)
std = max_std * torch.ones(plan_length, action_dim) # (4,2)
actions = torch.empty(plan_length, num_samples , action_dim)  # (4, 3, 2)

In [12]:
losses = []
elite_means = []
elite_stds = []

for _ in range(n_iters): 
    # actions[:, :, :] actions[:],actions[...] all works
    #                       (4, 1, 2)        + (4, 1, 2) * (4, 3, 2)
    actions[:, :] = mean.unsqueeze(1) + std.unsqueeze(1) * torch.randn(plan_length, num_samples, action_dim)

    #======== self.cost_function =========
    # why not just start with b a t shape rather than t b a shape in the first place ? 
     # sum_all_diffs = True [overriden by plan_cfg],
    cost = (torch.rand(num_samples)* torch.randint(1,3,(num_samples,))).unsqueeze(1)    # cost.shape = (num_samples=3, 1)
    
    losses.append(cost.min().item())

    # Get elite actions 
    elite_idxs = torch.topk(-cost.squeeze(1), num_elites, dim=0).indices
    # elite_loss.shape=(num_elites,1 ) | elite_actions.shape=(plan_length, num_elites, action.dim)
    elite_loss, elite_actions = cost[elite_idxs], actions[:, elite_idxs] 
    print(elite_loss)

    # record statistcs 
    elite_means.append(elite_loss.mean().item())
    elite_stds.append(elite_loss.std().item())

    min_cost = cost.min(0)[0]   # min_cost.shape=[1]
    score = torch.exp(temperature * (min_cost - elite_loss[:, 0]))   # score.shape = num_elites
    score/= score.sum(0)    # score.shape = num_elites 

    mean = torch.sum(score.unsqueeze(0).unsqueeze(2) * elite_actions, dim=1) / (score.sum(0) +1e-9)
    std =     torch.sqrt(
    torch.sum(
        score.unsqueeze(0).unsqueeze(2) 
        * ((elite_actions - mean.unsqueeze(1)) ** 2), 
        dim=1
        ) / (score.sum(0) + 1e-9) # this is useless, it comes from TD-MPC boilerplate
    )
    print(std.shape)
    # use the calculated mean and std for the next iteration 
    break 
     

tensor([[0.2943],
        [0.5485]])
torch.Size([4, 2])


In [ ]:
elite_loss = torch.tensor([[0.4636],[0.8133]])
score = torch.exp(temperature * (min_cost - elite_loss[:,0]))
score = score/score.sum(0)
mean = torch.sum(score.unsqueeze(0).unsqueeze(2) * elite_actions, dim=1)